# OSAI Project 1 — Colab v1 Baseline Reproduction

Pascal-VOC 21-class semantic segmentation, ResNet-50 + DeepLabV3+ (OS=16). main 브랜치 기준 v1 baseline 재현.

총 소요 시간 (T4): ~12-14h. Pro+ 단일 세션 안에 fit.

산출물 (Drive에 저장): `model_structure.onnx`, `submission_pred.zip`, `checkpoints/model.pth`.

## 사전 준비

1. Colab Pro+ + GPU 활성화 (런타임 → 런타임 유형 변경 → T4 또는 L4)
2. Drive에 폴더 생성: `MyDrive/osai/p1/input/test_public/` (1000장 jpg)
3. WandB API key 준비 (https://wandb.ai/authorize)

셀 순서대로 실행 또는 Run All.

## 1. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 설정 변수

필요시 수정. 기본값은 v1 baseline + Drive 표준 위치.

In [ ]:
REPO_URL = "https://github.com/geniemo/osai.git"
BRANCH   = "main"
CONFIG   = "src/config/colab.yaml"
DRIVE    = "/content/drive/MyDrive/osai/p1"

import os
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
print(f"Repo: {REPO_URL} (branch={BRANCH})")
print(f"Config: {CONFIG}")
print(f"Drive: {DRIVE}")
assert os.path.exists(f"{DRIVE}/input/test_public"), f"test_public/ not found at {DRIVE}/input/test_public"
n_imgs = len([f for f in os.listdir(f"{DRIVE}/input/test_public") if f.endswith(".jpg")])
print(f"test_public: {n_imgs} jpg files")

## 3. 저장소 clone

In [ ]:
%cd /content
!rm -rf osai
!git clone --branch {BRANCH} --depth 1 {REPO_URL} osai
%cd osai/p1

## 4. uv 설치 + 의존성 sync

PyTorch 2.7+ + CUDA 12.8 binary 자동 설치. ~3–5분.

In [ ]:
%cd /content/osai/p1
!pip install -q uv
!uv sync

## 5. GPU 확인

**기대**: Tesla T4 (capability 7,5) 또는 NVIDIA L4 (8,9). 다른 GPU면 spec 위반.

In [ ]:
!uv run python -c "\
import torch; \
print('Device:', torch.cuda.get_device_name(0)); \
print('Capability:', torch.cuda.get_device_capability(0)); \
print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB'); \
print('PyTorch:', torch.__version__)"

## 6. WandB 로그인

API key 입력 (https://wandb.ai/authorize 에서 복사). Colab Secrets에 저장돼 있으면 자동 사용.

In [ ]:
# Option A: 환경변수로 (Colab Secrets 사용 권장)
# from google.colab import userdata
# import os; os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

# Option B: interactive prompt (Run 후 key 붙여넣기)
!uv run wandb login

## 7. Test images 복사 (Drive → Colab)

Drive의 `osai/p1/input/test_public/` 폴더 → Colab `input/test_public/`. 빠른 inference 위해 복사 (symlink 대신).

In [ ]:
!mkdir -p input
!cp -r {DRIVE}/input/test_public input/
# Drive에 폴더로 올라가 있어서 복사만 (unzip 불필요)
# Colab local로 복사: ~30-60sec, 이후 inference 시 빠른 I/O
!ls input/test_public | wc -l   # 1000 expected
!ls input/test_public | head -3

## 8. 데이터 다운로드 (VOC + COCO)

VOC ~2GB (5분), COCO ~25GB (20–40분). 압축 해제 자동.

In [ ]:
!uv run python -m src.data.download --voc-root data/voc --coco-root data/coco

## 9. COCO mask cache 사전 생성

95K instance mask를 PNG로 1회 생성 (~10–15분). 학습 시 cache hit으로 빠름.

In [ ]:
!uv run python -m src.build_coco_masks --coco-root data/coco --num-workers 4

## 10. 학습 (Stage 1 + Stage 2 자동, ~10-12h)

ckpt가 Drive에 저장됨. 세션 끊기면 다시 시작 시 자동 resume.

In [ ]:
!uv run python -m src.train --config {CONFIG}

## 11. Validation mIoU 측정

최종 ckpt로 val mIoU + 21 class별 IoU 출력.

In [ ]:
!uv run python -m src.eval --config {CONFIG} --ckpt {DRIVE}/checkpoints/model.pth

## 12. TTA Validation mIoU (선택, 실제 inference 성능)

Multi-scale + hflip TTA로 측정. ~3–5분 (T4).

In [ ]:
!uv run python -m src.eval_tta --config {CONFIG} --ckpt {DRIVE}/checkpoints/model.pth

## 13. 추론 — test_public 1000장 (TTA)

input/test_public/*.jpg → output/pred_FINAL/*.png. ~5–10분 (T4).

In [ ]:
!mkdir -p output
!uv run python -m src.infer \
    --config {CONFIG} \
    --ckpt {DRIVE}/checkpoints/model.pth \
    --input input/test_public \
    --output output/pred_FINAL

## 14. ONNX export (가중치 제거, 채점용 ONNX)

`model_structure.onnx` 생성 (~0.3MB, ≤10MB 채점 한도).

In [ ]:
!uv run python -m src.export_onnx \
    --config {CONFIG} \
    --ckpt {DRIVE}/checkpoints/model.pth \
    --out {DRIVE}/model_structure.onnx

## 15. FLOPs 측정 (채점 기준 확인)

In [ ]:
!uv run python -m src.measure_flops --onnx {DRIVE}/model_structure.onnx

## 16. PNG zip 패키징 (채점용 PNG 제출)

1000 PNG (000.png~999.png), 픽셀 [0,20], 압축해제 ≤500MB 검증 + zip 생성.

In [ ]:
!uv run python -m src.package_submission \
    --pred output/pred_FINAL \
    --out {DRIVE}/submission_pred.zip

## 17. 결과물 확인

- `model_structure.onnx` (~0.3MB) — 채점 사이트 ONNX 업로드
- `submission_pred.zip` (~10-30MB) — 채점 사이트 PNG 업로드
- `checkpoints/model.pth` (~180MB) — 학교 사이트 zip 포함

In [ ]:
!ls -la {DRIVE}/
!ls -la {DRIVE}/checkpoints/

## 채점 사이트 업로드

- PNG zip: `MyDrive/osai/p1/submission_pred.zip` 다운 → PNG 슬롯
- ONNX: `MyDrive/osai/p1/model_structure.onnx` 다운 → ONNX 슬롯

## 학교 사이트 zip (로컬에서)

```bash
# Drive에서 model.pth 다운 후
cp ~/Downloads/model.pth p1/checkpoints/model.pth
# report.pdf 추가 후
zip -r p1/<학번>_project01.zip \
    p1/src p1/checkpoints/model.pth p1/submit \
    p1/<학번>_project01_report.pdf p1/pyproject.toml p1/README.md \
    -x '**/__pycache__/*' '**/.venv/*'
```